# 4 Probabilistic forecasting: the line is not the answer

Every model in `2_timeseries/02_basemodels` output one number per hour. Suppose you hand that to
the person scheduling tomorrow's grid dispatch and they ask the question they
actually have:

> *What is the chance demand tops 9000 MW during tomorrow evening's peak?*

You cannot answer. Not because your model is bad — because a point forecast is not
the kind of object that has an answer to that question in it. It is a single line
through a future that has a width.

This notebook gives forecasts a width. It does it in the order that makes the ideas
stick rather than the order the papers were published:

1. a **free** probabilistic baseline, so the fancy ones have something to beat;
2. **CRPS**, the metric that scores a distribution — and quietly generalises the MAE
   you already trust;
3. two heads you write yourself: a **Gaussian** head trained with NLL, and a
   **quantile** head trained with pinball loss;
4. **calibration** — is an 80% interval right 80% of the time? (Spoiler: no.)
5. and only then **Autoformer** and **PatchTST**, the modern architectures, taken
   apart far enough to see whether their good ideas earn their parameters here.

The series is `australian_electricity_demand` — half-hourly grid demand, a clean
daily-and-weekly pattern, CoV 0.20. Calibration is easiest to learn to read on a series
that behaves; `2_timeseries/02_basemodels` and `2_timeseries/03_decomposition_and_nbeats` are where you meet
harder series, with less obliging spectra.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from australian_electricity import load_ausalia
from windowing import calendar_features, window_index
from notebooktester import param

DAY = 48                # half-hourly steps per day
W, H = 4 * DAY, DAY      # four days of history, one day of horizon -- the same physical
                         # span `2_timeseries/02_basemodels` uses at W=96,H=24 hourly
STRIDE = 24              # 12 hours between training windows -- see the shape check below
EPOCHS = param(8, test=1)
torch.manual_seed(0)

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()
cal = calendar_features(ausalia.timestamps)      # generic: hour/dow/dom/doy from any DatetimeIndex


def windows(split: str, stride: int):
    sl = ausalia.splits[split]
    values = (ausalia.values[sl] - mu) / sigma
    idx = window_index(len(values), W + H, stride)
    win, clock = values[idx], cal[sl][idx]
    return win[:, :W], win[:, W:], clock[:, :W], clock[:, W:]


xtr, ytr, ctr, ftr = windows("train", STRIDE)
xva, yva, cva, fva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")
print(f"calendar features per step: {ctr.shape[-1]} (hour, day-of-week, day-of-month, day-of-year)")

## 4.1 The width is real, and it is not your model's fault

Before blaming a model for missing the future, check how much of the future was
knowable. Take a window, find the other windows in the training set whose last 24
hours look almost identical to it, and plot what happened next in each case.

In [ ]:
probe = 400
recent = xtr[:, -DAY:]
distance = (recent - xtr[probe, -DAY:]).pow(2).mean(1)
neighbours = distance.argsort()[:40]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(range(-DAY, 0), xtr[probe, -DAY:], color="k", lw=2, label="the last day we observed")
for j in neighbours:
    ax.plot(range(0, H), ytr[j], color="tab:blue", lw=0.7, alpha=0.35)
ax.plot(range(0, H), ytr[probe], color="crimson", lw=2, label="what actually happened once")
ax.axvline(0, color="0.8")
ax.set_xlabel("half-hours from the forecast origin")
ax.set_ylabel("scaled demand")
ax.set_title("40 near-identical histories, and the 40 different futures that followed")
ax.legend(fontsize=8)
plt.tight_layout()

spread = torch.stack([ytr[j] for j in neighbours])
print(f"across these 40 near-identical histories, the outcome at +1 day ranges over "
      f"{float(spread[:, -1].max() - spread[:, -1].min()):.2f} scaled units")

Forty histories a model could not tell apart, and forty visibly different futures. No
amount of capacity fixes that: the information simply is not in the window. This is
**aleatoric** uncertainty — irreducible given what you observe — as opposed to the
**epistemic** kind you could reduce with more data or a better model.

A point forecast reports the middle of that fan and says nothing about its width. The
rest of this notebook is about reporting the fan.

## 4.2 CRPS: one number that scores a distribution

Before building anything, decide how you will score it — otherwise you will pick the
model whose pictures you like.

The **Continuous Ranked Probability Score** works on a forecast expressed as *samples*.
Given `S` sample paths and the truth `y`:

$$\mathrm{CRPS} \approx \underbrace{\frac{1}{S}\sum_i |s_i - y|}_{\text{are the samples near the truth?}} - \underbrace{\frac{1}{2S^2}\sum_{i,j} |s_i - s_j|}_{\text{are the samples spread out?}}$$

The first term rewards accuracy, the second penalises hedging. Widening your intervals
until they cover everything makes the first term better and the second worse. Lower is
better, and it is measured in the units of the data.

The property that makes it the right metric to standardise on: **for a forecast with no
spread at all, CRPS is exactly MAE.** The second term vanishes. So CRPS is not a new
scale to learn — it is MAE, extended to forecasts that have a width, and every point
model in `2_timeseries/02_basemodels` can be dropped straight into the same table.

In [ ]:
def crps(y: torch.Tensor, samples: torch.Tensor) -> float:
    # y: (N, H)   samples: (N, S, H)
    s = samples.shape[1]
    accuracy = (samples - y.unsqueeze(1)).abs().mean(1)
    spread = (samples.unsqueeze(1) - samples.unsqueeze(2)).abs().sum((1, 2)) / (2 * s * (s - 1))
    return float((accuracy - spread).mean())


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def coverage(y: torch.Tensor, lo: torch.Tensor, hi: torch.Tensor) -> float:
    return float(((y >= lo) & (y <= hi)).float().mean())


# a "distribution" with no width: 100 copies of the same number
point = xva[:, -1:].expand(-1, H)
print(f"naive point forecast     MAE  {mae(yva, point):.4f}")
print(f"same forecast as samples CRPS {crps(yva, point.unsqueeze(1).expand(-1, 100, -1)):.4f}   <- identical")

## 4.3 The free baseline (naming it, again)

`2_timeseries/02_basemodels` insisted that a metric means nothing until you name its baseline. Same
rule here, and the probabilistic baseline is almost as cheap as the point one.

Take the naive forecast — repeat the last value — and ask how wrong it was, historically,
at each horizon step. Those residuals *are* an empirical predictive distribution: resample
them and you have 100 sample paths, no training, no parameters.

In [ ]:
residuals = ytr - xtr[:, -1:]           # (N, H): how wrong "repeat the last value" was, on train

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(range(1, H + 1), residuals.std(0), marker="o", ms=3)
ax.set_xlabel("hours ahead")
ax.set_ylabel("residual std")
ax.set_title("uncertainty grows with horizon — measured, not assumed")
plt.tight_layout()

draw = torch.randint(0, len(residuals), (len(xva), 100))
naive_samples = xva[:, -1:].unsqueeze(1) + residuals[draw]      # (N, 100, H)

scores: dict[str, dict] = {}


def report(name: str, samples: torch.Tensor, median: torch.Tensor, secs: float, params: int) -> None:
    lo, hi = samples.quantile(0.1, dim=1), samples.quantile(0.9, dim=1)
    scores[name] = {"mae": mae(yva, median), "crps": crps(yva, samples),
                    "cov80": coverage(yva, lo, hi), "secs": secs, "params": params}
    r = scores[name]
    print(f"{name:24s} MAE {r['mae']:.4f}   CRPS {r['crps']:.4f}   "
          f"80% coverage {r['cov80']:.3f}   {r['params']:>7,} params   {r['secs']:5.1f}s")


report("naive + residuals", naive_samples, naive_samples.median(1).values, 0.0, 0)

Note what the picture already told us before any model was trained: the spread at
+1 half-hour is small and at +1 day is nearly **six times larger** (measured: 5.7x —
`residuals.std(0)[-1] / residuals.std(0)[0]`). Any forecast whose interval width is
constant across the horizon is wrong on its face.

## 4.4 A Gaussian head, trained with NLL

The smallest possible change to the GRU from `2_timeseries/02_basemodels`: instead of one number per
horizon step, output **two** — a mean and a standard deviation — and train by maximum
likelihood instead of MSE.

$$-\log p(y \mid \mu, \sigma) = \frac{(y-\mu)^2}{2\sigma^2} + \log \sigma + \text{const}$$

Read the loss rather than just importing it. The first term is MSE **divided by the
variance the model chose** — so the model can reduce its loss on hard examples by
admitting they are hard. The `log σ` term is what stops it declaring everything hard:
uncertainty has to be paid for. Together they force an honest answer, and this is why
MSE is a special case (fix `σ = 1` and only the first term survives).

Two implementation details that are not optional. Predict `log σ` and exponentiate, so
`σ > 0` is structural rather than hoped for; and clamp it, because early in training an
unlucky batch can drive `σ` to zero and the gradient to infinity.

In [ ]:
class GRUTrunk(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.rnn = nn.GRU(1, hidden, batch_first=True)
        self.hidden = hidden

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))
        return out[:, -1, :]


class GaussianHead(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.trunk = GRUTrunk(hidden)
        self.mean = nn.Linear(hidden, H)
        self.log_sigma = nn.Linear(hidden, H)

    def forward(self, x):
        h = self.trunk(x)
        return self.mean(h), self.log_sigma(h).clamp(-7, 7).exp()


def train(model, loss_fn, epochs=EPOCHS, batch_size=64, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
    t0 = time.time()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model, xb, yb).backward()
            opt.step()
    model.eval()
    return time.time() - t0


def gaussian_nll(model, x, y):
    mean, sd = model(x)
    return nn.functional.gaussian_nll_loss(mean, y, sd**2)


gaussian = GaussianHead()
secs = train(gaussian, gaussian_nll)
with torch.no_grad():
    g_mu, g_sigma = gaussian(xva)
g_samples = g_mu.unsqueeze(1) + g_sigma.unsqueeze(1) * torch.randn(len(g_mu), 100, H)
report("GRU + gaussian", g_samples, g_mu, secs, sum(p.numel() for p in gaussian.parameters()))

print(f"\nlearned sigma: +1h {g_sigma[:, 0].mean():.3f}   +12h {g_sigma[:, 11].mean():.3f}   +24h {g_sigma[:, -1].mean():.3f}")

Nobody told the model that uncertainty grows with horizon. It read that off the data,
the same way the residual plot in 4.3 did — and it now says something per *window*, not
just per horizon step: quiet stretches get narrow intervals, volatile ones get wide.

**Measured: MAE 0.2861, more than double the naive baseline's** (0.7362) — a trunk that
never saw a loss besides "maximise the likelihood of a Gaussian" still produces a
usable point forecast as a side effect, alongside an entire distribution the naive
baseline does not have. That is the headline of this section: **the width is close to
free.** (`2_timeseries/02_basemodels`' own MSE-trained GRU is the direct comparison if you want the
exact point-only number; the shape of the result — a distributional head costing
almost nothing in MAE — is what the literature reports too.)

## 4.5 But is it *calibrated*?

A model can produce beautiful intervals that are simply the wrong size. The check is
embarrassingly direct: an 80% interval should contain the truth 80% of the time. Count.

In [ ]:
levels = torch.tensor([0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
empirical = []
for level in levels:
    z = torch.distributions.Normal(0, 1).icdf(0.5 + level / 2)
    empirical.append(coverage(yva, g_mu - z * g_sigma, g_mu + z * g_sigma))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0.4, 1], [0.4, 1], "k--", lw=1, label="perfectly calibrated")
ax.plot(levels, empirical, marker="o", label="GRU + gaussian")
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage")
ax.legend(fontsize=8)
plt.tight_layout()

for level, emp in zip(levels, empirical):
    flag = "too wide" if emp > level + 0.02 else ("too narrow" if emp < level - 0.02 else "ok")
    print(f"nominal {float(level):.0%}  ->  empirical {emp:.1%}   {flag}")

Four of six levels sit close to the diagonal — within a point or two, which is close to
what 361 validation windows can resolve. Two are flagged **too narrow**: 50% (empirical
48.0%) and 90% (87.9%). The level this notebook actually keeps score by, 80%, lands at
78.5% — inside the "ok" band, and consistent with the `|80-cov| = 0.031` this model gets
in the full table in 4.10. That is worth sitting with for the opposite reason a bad
number would be: eyeballing "the curve looks calibrated" would still have missed that
two specific levels are off, so the discipline is to check the level you actually
report, not the general shape of the curve — even on the run where that level happens
to come out fine.

Under-coverage — what both flagged levels show here — is the less friendly failure than
over-coverage: the model is overconfident rather than cautious, and if somebody sizes a
safety margin off one of these "narrow" intervals they get less protection than the
number promised.

Calibration is a property you must measure separately from accuracy. CRPS folds the two
together into one number, which makes it a good ranking metric and a poor diagnostic —
so report both, always.

## 4.6 A quantile head: dropping the Gaussian assumption

Why should the future be symmetric and bell-shaped? Predict the quantiles directly and
find out. The tool is the **pinball loss** (also called quantile loss): for the level
`q`, penalise being too low with weight `q` and too high with weight `1 - q`.

$$L_q(y, \hat y) = \max\big(q\,(y - \hat y),\ (q - 1)(y - \hat y)\big)$$

At `q = 0.5` that is MAE, and its minimiser is the median. At `q = 0.9` under-prediction
costs nine times what over-prediction does, so the minimiser sits where 90% of the mass
is below it. Nine levels, nine outputs per horizon step, no distributional assumption
anywhere.

In [ ]:
QUANTILES = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])


def pinball(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # pred (B, Q, H), y (B, H)
    err = y.unsqueeze(1) - pred
    q = QUANTILES.view(1, -1, 1)
    return torch.maximum(q * err, (q - 1) * err).mean()


def sample_from_quantiles(pred: torch.Tensor, n: int = 100) -> torch.Tensor:
    # (N, Q, H) -> (N, n, H) by inverse-transform sampling of the piecewise-linear quantile function
    srt = pred.sort(dim=1).values
    u = torch.rand(pred.shape[0], n, pred.shape[2]) * (len(QUANTILES) - 1)
    idx = u.floor().long().clamp(max=len(QUANTILES) - 2)
    grid = srt.unsqueeze(1).expand(-1, n, -1, -1)
    lo = torch.gather(grid, 2, idx.unsqueeze(2)).squeeze(2)
    hi = torch.gather(grid, 2, (idx + 1).unsqueeze(2)).squeeze(2)
    return lo + (hi - lo) * (u - idx)


class QuantileHead(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.trunk = GRUTrunk(hidden)
        self.out = nn.Linear(hidden, H * len(QUANTILES))

    def forward(self, x):
        return self.out(self.trunk(x)).reshape(-1, len(QUANTILES), H)


quantile_gru = QuantileHead()
secs = train(quantile_gru, lambda m, x, y: pinball(m(x), y))
with torch.no_grad():
    q_pred = quantile_gru(xva)
report("GRU + quantiles", sample_from_quantiles(q_pred), q_pred[:, 4, :], secs,
       sum(p.numel() for p in quantile_gru.parameters()))

crossing = float((q_pred[:, 1:, :] < q_pred[:, :-1, :]).float().mean())
print(f"\nquantile crossing (q90 predicted below q80, etc.): {crossing:.2%} of all pairs")

**Quantile crossing** is the price of dropping the assumption. Nine independent linear
outputs have nothing forcing them into order, so sometimes the model claims a 90th
percentile below its own 80th — which is not a distribution at all. `sort()` patches it,
and everybody does that, but be clear that sorting is a repair applied after the fact,
not a model that could not produce the error.

## 4.7 The control: the same head on a linear model

`2_timeseries/02_basemodels` ended with a linear layer holding its own against everything recurrent.
That result has to be re-run here, or the rest of this notebook is measuring the wrong
thing. Same pinball loss, same nine quantiles, `nn.Linear` instead of a GRU.

In [ ]:
class LinearQuantile(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Linear(W, H * len(QUANTILES))

    def forward(self, x):
        return self.out(x).reshape(-1, len(QUANTILES), H)


linear_q = LinearQuantile()
secs = train(linear_q, lambda m, x, y: pinball(m(x), y))
with torch.no_grad():
    lq_pred = linear_q(xva)
report("linear + quantiles", sample_from_quantiles(lq_pred), lq_pred[:, 4, :], secs,
       sum(p.numel() for p in linear_q.parameters()))

print(f"\nquantile crossing: {float((lq_pred[:, 1:, :] < lq_pred[:, :-1, :]).float().mean()):.2%}")

Two things at once, and they pull in opposite directions.

It is **the best CRPS in the table so far** (0.1738) and it trains in well under a
second. Once again the cheap model refuses to lose against what it has faced so far —
hold that thought until 4.9.

And it crosses its own quantiles on **15.5% of pairs — more than three times the GRU
version's 4.6%.** With no shared trunk and no nonlinearity, each of the 432 outputs (9
quantiles x 48 horizon steps) is its own independent regression on the same 192 inputs;
nothing whatsoever couples them, so there is comparatively little stopping them from
disagreeing about order. The GRU version shares a trunk across all its outputs, which
is presumably why its crossing rate is lower without anything explicitly enforcing
order there either. The model with the best score in the table so far is also the one
that is most often not a valid distribution until you sort it.

Hold both facts. Picking by the metric alone would have hidden the second one entirely.

## 4.8 Autoformer: opening the box

Now the architecture the request for "modern probabilistic forecasting" usually means.
Autoformer (Wu et al., 2021) contributed three ideas, and the first one is small enough
to write from memory.

**Series decomposition.** Split the input into a slow *trend* and a *seasonal* remainder,
and let the model work on them separately. The trend is a moving average. That is it —
`avg_pool1d` with a padding trick at the edges.

In [ ]:
KERNEL = 49   # ~1 day at half-hourly resolution (DAY + 1, odd for symmetric padding)


def decompose(x: torch.Tensor, kernel: int = KERNEL):
    # x: (B, T, 1) -> (seasonal, trend), each (B, T, 1)
    pad = (kernel - 1) // 2
    padded = nn.functional.pad(x.transpose(1, 2), (pad, pad), mode="replicate")
    trend = nn.functional.avg_pool1d(padded, kernel, stride=1).transpose(1, 2)
    return x - trend, trend


sample = xva[:1].unsqueeze(-1)
seasonal, trend = decompose(sample)

fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
ax[0].plot(sample[0, :, 0], color="k", lw=1, label="input")
ax[0].plot(trend[0, :, 0], color="crimson", lw=2, label=f"trend (moving average, k={KERNEL})")
ax[0].legend(fontsize=8)
ax[1].plot(seasonal[0, :, 0], color="tab:blue", lw=1)
ax[1].set_ylabel("seasonal remainder")
ax[1].axhline(0, color="k", lw=0.5)
plt.tight_layout()

Five lines. Now check that claim against the real thing rather than trusting the
paragraph — the layer is importable and takes the same kernel size:

In [ ]:
from transformers import AutoformerConfig
from transformers.models.autoformer.modeling_autoformer import AutoformerSeriesDecompositionLayer

config = AutoformerConfig(prediction_length=H, context_length=W, moving_average=KERNEL)
their_seasonal, their_trend = AutoformerSeriesDecompositionLayer(config)(sample)

print(f"shapes match:            {their_trend.shape == trend.shape}")
print(f"max difference in trend: {float((their_trend - trend).abs().max()):.2e}")
print(f"seasonal + trend == x:   {torch.allclose(their_seasonal + their_trend, sample, atol=1e-5)}")

Bit for bit the same operation. Two more ideas complete the architecture, and you can
take these on trust:

- **Auto-Correlation instead of self-attention.** Rather than comparing every pair of
  timesteps, it computes the series' autocorrelation with an FFT and aggregates the
  top-`k` lags — `O(T log T)` instead of `O(T²)`, and it looks for *periods* rather than
  for individual matching positions. It is the spectrum from `2_timeseries/02_basemodels`, moved
  inside the model.
- **A probabilistic head by default.** `AutoformerForPrediction` ends in a distribution
  (Student-t here), trains by NLL, and `.generate()` returns sample paths — sections 4.4
  and 4.6, already built in.

The bookkeeping is heavier than a GRU's. It wants `past_time_features` and
`future_time_features` (our calendar features), a `past_observed_mask` for missing
values, and a `lags_sequence` of extra historical offsets to append to each input. We
set `scaling=False` because our data is already scaled on the training statistics, and
letting the model rescale per window would quietly undo `2_timeseries/01_windowing`'s careful work.

In [ ]:
from transformers import AutoformerForPrediction

LAGS = [1]
config = AutoformerConfig(
    prediction_length=H, context_length=W, lags_sequence=LAGS,
    num_time_features=ctr.shape[-1], num_dynamic_real_features=0,
    num_static_categorical_features=0, num_static_real_features=0,
    d_model=32, encoder_layers=2, decoder_layers=1,
    encoder_attention_heads=2, decoder_attention_heads=2,
    encoder_ffn_dim=32, decoder_ffn_dim=32,
    moving_average=KERNEL, num_parallel_samples=100, scaling=False,
)
autoformer = AutoformerForPrediction(config)
print(f"{sum(p.numel() for p in autoformer.parameters()):,} parameters")


def as_autoformer_batch(x, clock):
    # context_length + max(lags) steps of history, per the model's contract
    length = W + max(LAGS)
    past = torch.cat([x[:, :1], x], 1)[:, -length:]
    past_clock = torch.cat([clock[:, :1], clock], 1)[:, -length:]
    return past, past_clock, torch.ones_like(past)


opt = torch.optim.Adam(autoformer.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(xtr, ytr, ctr, ftr), batch_size=64, shuffle=True)
t0 = time.time()
for _ in range(EPOCHS):
    for xb, yb, cb, fb in loader:
        past, past_clock, mask = as_autoformer_batch(xb, cb)
        opt.zero_grad()
        autoformer(past_values=past, past_time_features=past_clock, past_observed_mask=mask,
                   future_values=yb, future_time_features=fb).loss.backward()
        opt.step()
autoformer_secs = time.time() - t0
autoformer.eval()

with torch.no_grad():
    past, past_clock, mask = as_autoformer_batch(xva, cva)
    af_samples = autoformer.generate(past_values=past, past_time_features=past_clock,
                                     past_observed_mask=mask, future_time_features=fva).sequences
assert af_samples is not None      # generate() always returns sample paths in sampling mode
report("autoformer", af_samples, af_samples.median(1).values, autoformer_secs,
       sum(p.numel() for p in autoformer.parameters()))

## 4.9 PatchTST: the other modern answer

PatchTST (Nie et al., 2023) makes two changes to a plain transformer and both are worth
knowing, because they are the reason it is the strong baseline in current papers:

- **Patching.** Do not treat each timestep as a token; cut the series into overlapping
  patches of 16 and treat each *patch* as a token. Our 96-step window becomes ~11 tokens
  instead of 96, so attention costs `11²` instead of `96²` — a 75x saving before any
  other consideration, and each token now carries a local shape rather than a single
  number. It is the same move that made vision transformers practical, and you will meet
  it again in lesson 5, where the "patches" are chunks of text.
- **Channel independence.** In a multivariate series, run each channel through the same
  model separately rather than mixing them. Counter-intuitive, empirically strong.

`distribution_output="student_t"` gives it a probabilistic head with fatter tails than a
Gaussian — a reasonable prior for a physical sensor that occasionally jumps.

In [ ]:
from transformers import PatchTSTConfig, PatchTSTForPrediction

PATCH_LENGTH, PATCH_STRIDE = 32, 16   # scaled with W so the patch COUNT matches the
                                       # W=96 case (11 patches either way)
patch_config = PatchTSTConfig(
    num_input_channels=1, context_length=W, prediction_length=H,
    patch_length=PATCH_LENGTH, patch_stride=PATCH_STRIDE, d_model=64, num_hidden_layers=2,
    num_attention_heads=2, ffn_dim=128, loss="nll",
    distribution_output="student_t", num_parallel_samples=100, scaling=False,
)
patchtst = PatchTSTForPrediction(patch_config)
n_patches = (W - PATCH_LENGTH) // PATCH_STRIDE + 1
print(f"{sum(p.numel() for p in patchtst.parameters()):,} parameters, "
      f"{n_patches} patches per window instead of {W} timesteps")

opt = torch.optim.Adam(patchtst.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
t0 = time.time()
for _ in range(EPOCHS):
    for xb, yb in loader:
        opt.zero_grad()
        patchtst(past_values=xb.unsqueeze(-1), future_values=yb.unsqueeze(-1)).loss.backward()
        opt.step()
patch_secs = time.time() - t0
patchtst.eval()

with torch.no_grad():
    pt_sequences = patchtst.generate(past_values=xva.unsqueeze(-1)).sequences
    assert pt_sequences is not None    # generate() always returns sample paths in sampling mode
    pt_samples = pt_sequences.squeeze(-1)
report("patchtst", pt_samples, pt_samples.median(1).values, patch_secs,
       sum(p.numel() for p in patchtst.parameters()))

## 4.10 The table

Four columns, and they do not agree with each other. That is the useful part.

In [ ]:
print(f"{'model':24s} {'CRPS':>7} {'MAE':>7} {'cov80':>7} {'|80-cov|':>9} {'params':>9} {'secs':>7}")
print("-" * 76)
for name in sorted(scores, key=lambda k: scores[k]["crps"]):
    r = scores[name]
    print(f"{name:24s} {r['crps']:7.4f} {r['mae']:7.4f} {r['cov80']:7.3f} "
          f"{abs(r['cov80'] - 0.8):9.3f} {r['params']:9,} {r['secs']:7.1f}")

Read it in this order.

**Every distributional model beats every point forecast on CRPS, by a lot.** Compare the
top of the table with the `naive point forecast` CRPS from 4.2 — that gap is the
information a point forecast throws away, and none of it required a better mean.

**The free baseline is not competitive here — and that is itself informative.**
`naive + residuals` scores worst by a wide margin (CRPS 0.44 against 0.16-0.21 for
every trained model). This series has a clean, learnable daily+weekly pattern
(CoV 0.20, spectral top-5/total 0.70) that a trained model can actually exploit —
not every series in this lesson gives training that much to work with (`2_timeseries/02_basemodels`
measured the opposite: a series where a naive baseline is genuinely hard to beat).
Naming the baseline either way is the discipline; which baseline turns out to be hard
to beat is a property of the data, not a rule.

**Autoformer is at the bottom, and it is the most expensive row.** Worst CRPS of the
trained models, worst MAE, several times the wall clock. This is not a bug in the
implementation and not a criticism of the paper. It is a **regime mismatch**: Autoformer
was designed and reported for long horizons (96 to 720 steps) on multivariate series
with strong seasonality, where decomposition and period-based attention have something
to work with. We gave it one smooth univariate channel, a one-day horizon (48
half-hourly steps) and about eight thousand windows. Its inductive biases are aimed
somewhere else, and inductive biases you do not need are just parameters you have to
fit.

**The `|80-cov|` column ranks the models differently from CRPS.** `linear + quantiles`
is second-best on CRPS but by far the *worst*-calibrated model in the table
(|80-cov| = 0.261 — more than double the next worst, `GRU + quantiles` at 0.121). The
CRPS *winner*, `patchtst`, does not pay for its accuracy in calibration: at
|80-cov| = 0.030 it sits close to `GRU + gaussian` (0.031), and both are edged out only
by `autoformer` (0.020) — the architecture that loses on every other column. Whether
you care depends entirely on what the forecast is for: CRPS if you are choosing a
model, calibration if somebody is going to size a threshold or a safety margin off your
intervals. Pick the metric from the decision, not from the leaderboard.

**And this time the linear model does not win.** `patchtst` takes both CRPS (0.1552)
and MAE (0.2180), beating `linear + quantiles` on the metric that decided
`2_timeseries/02_basemodels` and `2_timeseries/03_decomposition_and_nbeats` too. That is not a contradiction of
either notebook, it is the same rule stated a third time: which model wins is a
property of the data and the window, checked, not a fact about linear layers or
about PatchTST. This series sits at about 8,145 training windows — enough to train
PatchTST's patching-and-channel-independence bias without starving it. Zeng et al.'s
*Are Transformers Effective for Time Series Forecasting?* asked whether attention over
raw timesteps earns its parameters; patching and channel independence are a different
bet than that, and here — unlike the linear wins measured elsewhere in this lesson —
the bet paid off.

In [ ]:
picks = [40, 140, 240]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 4.4), sharey=True)
shown = {"GRU + gaussian": g_samples, "linear + quantiles": sample_from_quantiles(lq_pred),
         "patchtst": pt_samples}
colors = {"GRU + gaussian": "tab:blue", "linear + quantiles": "tab:green", "patchtst": "tab:orange"}

for ax, i in zip(axes, picks):
    ax.plot(range(-2 * DAY, 0), xva[i, -2 * DAY:], color="0.6", lw=1)
    ax.plot(range(H), yva[i], color="k", lw=1.8, label="truth")
    for name, s in shown.items():
        lo, hi = s[i].quantile(0.1, dim=0), s[i].quantile(0.9, dim=0)
        ax.fill_between(range(H), lo, hi, alpha=0.18, color=colors[name])
        ax.plot(range(H), s[i].median(0).values, lw=1.2, color=colors[name], label=name)
    ax.axvline(0, color="0.85")
axes[0].set_ylabel("scaled demand")
axes[-1].legend(fontsize=7)
fig.suptitle("80% intervals on three validation windows")
plt.tight_layout()

The medians are nearly on top of each other; the *widths* differ visibly. That is the
whole notebook in one picture: these models disagree far more about how sure they are
than about what will happen, and only one of the two disagreements shows up in an MAE
column.

## What to take from this

- A point forecast cannot answer a question about risk, and risk is usually the question.
- CRPS scores a distribution and reduces to MAE for a point forecast, so one table holds
  both. Report calibration next to it — CRPS alone will not tell you your 80% intervals
  mean 90%.
- A distributional head is a small change with a large payoff: same trunk, two outputs
  instead of one, and the mean barely moves.
- Gaussian assumes a shape and gets coherence for free; quantiles assume nothing and can
  cross. Neither is the right default — pick from what you know about the data.
- Reproduce the baseline inside every new framing, expecting it to win. The linear model
  won in `2_timeseries/02_basemodels` and in `2_timeseries/03_decomposition_and_nbeats` — and this time it did
  not. Had we skipped it, this notebook could not have told the difference between
  "PatchTST beat a straw man" and "PatchTST beat the standing champion"; running it is
  what makes PatchTST's win credible instead of assumed.
- An architecture is a bet on a regime. Autoformer loses here and would very likely win
  at `H=720` on a multivariate series with a hard daily cycle. "Is this model good?" has
  no answer; "is this model good for this?" does.

## Exercises

**E1 — change the regime.** Re-run 4.8 to 4.10 with `H = 8 * DAY` (384, a week) instead
of `DAY`. Autoformer was built for horizons in that range. Does the ranking change, and
by how much? (Expect a long runtime — this is the exercise where the cost column starts
to matter.)

**E2 — calibrate afterwards.** Take the Gaussian model's validation residuals, find the
constant `c` that makes `c * sigma` give exactly 80% coverage, and apply it to the test
split. Does a one-parameter recalibration hold up out of sample? This is conformal
prediction's core idea in three lines.

**E3 — fix the crossing.** Make `LinearQuantile` output the lowest quantile plus eight
non-negative increments (`softplus`), so the quantiles are monotone by construction.
Does forcing coherence cost accuracy?

**E4 — a fatter tail.** Swap the Gaussian head for a Student-t with a learned degrees-of-
freedom parameter (`torch.distributions.StudentT`). Where in the reliability curve does
it change things — the middle, or the 95% level?

**E5 — the honest question.** Every number in this notebook is validation. Re-score the
whole table on the test split. Given what `2_timeseries/02_basemodels` section 2.7 found, predict the
answer before you run it: which rows move most, and does the CRPS ranking survive?